<a href="https://colab.research.google.com/github/nhuvtq87/AAI2025/blob/2026fall/ML_Coding_Exercise.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# ---------------------------------------------------------------
# PART 1: House Price Prediction (Linear Regression)
# Data source: house_prediction_price.csv -- the King County House Sales dataset
# (Kaggle: https://www.kaggle.com/datasets/harlfoxem/housesalesprediction),
# 21,613 real home sales in King County, WA (2014-2015) with price, square_footage, and location)
# ---------------------------------------------------------------
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, r2_score
import matplotlib.pyplot as plt

df1 = pd.read_excel('/content/house_prediction_price_with_location.xlsx')
df1 = df1.dropna(subset=['price', 'square_footage', 'location'])

# Features and target
X1 = df1[['square_footage', 'location']]
y1 = df1['price']

# Preprocessing: one-hot encode location, pass square_footage through
preprocessor1 = ColumnTransformer(transformers=[('location', OneHotEncoder(sparse_output=False, handle_unknown='ignore'), ['location'])], remainder='passthrough')
# Create pipeline with preprocessing and model
house_model = Pipeline(steps=[('preprocessor', preprocessor1),('regressor', LinearRegression())])
# Split data
X1_train, X1_test, y1_train, y1_test = train_test_split(X1, y1, test_size=0.2, random_state=42)
# Train model
house_model.fit(X1_train, y1_train)

# Predict price for a 2000 sq ft house in Downtown
new_house = pd.DataFrame({'square_footage': [2000], 'location': ['Downtown']})
predicted_price = house_model.predict(new_house)
print(f"Predicted price for a 2000 sq ft house in Downtown: ${predicted_price[0]:,.2f}")

# Display model coefficients
feature_names1 = (house_model.named_steps['preprocessor']
                   .named_transformers_['location']
                   .get_feature_names_out(['location'])).tolist() + ['square_footage']
coefficients1 = house_model.named_steps['regressor'].coef_

print("\nModel Coefficients:")
for feature, coef in zip(feature_names1, coefficients1):
    print(f"{feature}: {coef:,.2f}")

Predicted price for a 2000 sq ft house in Downtown: $514,196.23

Model Coefficients:
location_Downtown: -3,779.24
location_Rural: 316.83
location_Suburb: 3,462.41
square_footage: 281.22


In [3]:
# The interpretation generated from the actual fitted coefficients
sqft_coef = coefficients1[-1]
location_coefs = dict(zip(feature_names1[:-1], coefficients1[:-1]))
best_loc = max(location_coefs, key=location_coefs.get)
worst_loc = min(location_coefs, key=location_coefs.get)

print("INTERPRETATION:")
print(f"- Square footage: each additional square foot is associated with about "
      f"${sqft_coef:,.2f} more in predicted price, holding location constant.")
print(f"- Location: holding square footage constant, '{best_loc.split('_')[-1]}' "
      f"commands the highest price premium (coef = {location_coefs[best_loc]:,.2f}), "
      f"while '{worst_loc.split('_')[-1]}' has the lowest (coef = {location_coefs[worst_loc]:,.2f}). "
      f"These differences reflect demand/desirability effects captured by the one-hot "
      f"encoded location feature, separate from the size-driven price increase above.")

INTERPRETATION:
- Square footage: each additional square foot is associated with about $281.22 more in predicted price, holding location constant.
- Location: holding square footage constant, 'Suburb' commands the highest price premium (coef = 3,462.41), while 'Downtown' has the lowest (coef = -3,779.24). These differences reflect demand/desirability effects captured by the one-hot encoded location feature, separate from the size-driven price increase above.


## Part 2: Customer Churn Prediction (Logistic Regression)

In [5]:
# ---------------------------------------------------------------
# PART 2: Customer Churn Prediction (Logistic Regression)
# Data source: synthetically generated dataset (n=300) with realistic
# feature distributions and a built-in churn-risk relationship
# (higher customer service calls / lower usage -> higher churn probability),
# created for this assignment since no proprietary churn dataset was available.
# ---------------------------------------------------------------
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# Load dataset
df2 = pd.read_csv("/content/sample_data/WA_Fn-UseC_-Telco-Customer-Churn.csv")

# Clean target and numeric columns
df2["TotalCharges"] = pd.to_numeric(
    df2["TotalCharges"], errors="coerce"
).fillna(0)
df2["Churn"] = df2["Churn"].map({"Yes": 1, "No": 0})

# Define feature columns and target
numeric_features = ["tenure", "MonthlyCharges", "TotalCharges"]
categorical_features = [
    "gender",
    "SeniorCitizen",
    "Partner",
    "Dependents",
    "PhoneService",
    "MultipleLines",
    "InternetService",
    "OnlineSecurity",
    "OnlineBackup",
    "DeviceProtection",
    "TechSupport",
    "StreamingTV",
    "StreamingMovies",
    "Contract",
    "PaperlessBilling",
    "PaymentMethod",
]

# Features and target
X2 = df2[numeric_features + categorical_features]
y2 = df2["Churn"]

# Preprocessing: Scale numerical features and one-hot encode categorical features
preprocessor2 = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_features),
        (
            "cat",
            OneHotEncoder(drop="first", sparse_output=False),
            categorical_features,
        ),
    ]
)

# Create pipeline with preprocessing and model
churn_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor2),
        ("classifier", LogisticRegression(random_state=42, max_iter=1000)),
    ]
)

# Split data
X2_train, X2_test, y2_train, y2_test = train_test_split(
    X2, y2, test_size=0.2, random_state=42
)

# Train model
churn_model.fit(X2_train, y2_train)

# Predict churn probability for a new customer
new_customer = pd.DataFrame(
    [
        {
            "tenure": 1,
            "MonthlyCharges": 70.35,
            "TotalCharges": 70.35,
            "gender": "Female",
            "SeniorCitizen": 0,
            "Partner": "No",
            "Dependents": "No",
            "PhoneService": "Yes",
            "MultipleLines": "No",
            "InternetService": "Fiber optic",
            "OnlineSecurity": "No",
            "OnlineBackup": "No",
            "DeviceProtection": "No",
            "TechSupport": "No",
            "StreamingTV": "No",
            "StreamingMovies": "No",
            "Contract": "Month-to-month",
            "PaperlessBilling": "Yes",
            "PaymentMethod": "Electronic check",
        }
    ]
)

churn_probability = churn_model.predict_proba(new_customer)[0][1]
# Probability of churn (class 1)

# Classify based on threshold (0.5)
threshold = 0.5
churn_prediction = 1 if churn_probability > threshold else 0

print(f"Churn Probability for new customer: {churn_probability:.2f}")
print(f"Churn Prediction (1 = churn, 0 = no churn): {churn_prediction}")

# Display model coefficients
feature_names2 = (
    numeric_features
    + churn_model.named_steps["preprocessor"]
    .named_transformers_["cat"]
    .get_feature_names_out(categorical_features)
    .tolist()
)
coefficients2 = churn_model.named_steps["classifier"].coef_[0]

print("\nModel Coefficients:")
for feature, coef in zip(feature_names2, coefficients2):
    print(f"{feature}: {coef:.4f}")

Churn Probability for new customer: 0.70
Churn Prediction (1 = churn, 0 = no churn): 1

Model Coefficients:
tenure: -1.3638
MonthlyCharges: -0.3351
TotalCharges: 0.6612
gender_Male: -0.0514
SeniorCitizen_1: 0.1609
Partner_Yes: 0.0541
Dependents_Yes: -0.1595
PhoneService_Yes: -0.5150
MultipleLines_No phone service: -0.0824
MultipleLines_Yes: 0.2899
InternetService_Fiber optic: 1.0006
InternetService_No: -0.1499
OnlineSecurity_No internet service: -0.1499
OnlineSecurity_Yes: -0.4006
OnlineBackup_No internet service: -0.1499
OnlineBackup_Yes: -0.1432
DeviceProtection_No internet service: -0.1499
DeviceProtection_Yes: 0.0066
TechSupport_No internet service: -0.1499
TechSupport_Yes: -0.3190
StreamingTV_No internet service: -0.1499
StreamingTV_Yes: 0.2699
StreamingMovies_No internet service: -0.1499
StreamingMovies_Yes: 0.3673
Contract_One year: -0.6339
Contract_Two year: -1.3969
PaperlessBilling_Yes: 0.3320
PaymentMethod_Credit card (automatic): -0.0879
PaymentMethod_Electronic check: 0.320